# Football schedules, lottery demand, and Sauvie Island harvest

This notebook compares Oregon Ducks, Oregon State Beavers, and Seattle Seahawks game dates with Sauvie Island first-choice reservation applications and harvest records for the 2021-22 through 2025-26 hunting seasons.

The chart uses a **same-season, same-weekday non-game baseline** for each team. This reduces the large Saturday/Sunday scheduling bias, but the results remain observational and are not causal.

In [ ]:
# Load one row per hunt date and build weekday-matched comparison groups.
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "processed" / "duckit_oregon.duckdb"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
con = duckdb.connect(str(DB_PATH), read_only=True)
hunt_days = con.sql("SELECT * FROM hunt_day_sports_comparison").df()

teams = [
    ("Oregon", "oregon_game"),
    ("Oregon State", "oregon_state_game"),
    ("Seahawks", "seahawks_game"),
]
total_columns = [
    "first_choice_applicants",
    "reservations_available",
    "ducks",
    "hunters",
]
comparison_rows = []

for team, flag in teams:
    game_days = hunt_days.loc[hunt_days[flag]].copy()
    actual = game_days[total_columns].sum().astype(float)
    baseline = pd.Series(0.0, index=total_columns)
    control_indexes = set()

    # Match the game-day mix by hunting season and weekday. Each stratum's
    # non-game average is weighted by the number of game dates in that stratum.
    for (season, weekday), games in game_days.groupby(["season", "weekday"]):
        controls = hunt_days.loc[
            (hunt_days["season"] == season)
            & (hunt_days["weekday"] == weekday)
            & (~hunt_days[flag])
        ]
        if controls.empty:
            raise ValueError(f"No controls for {team}, {season}, {weekday}")
        baseline += controls[total_columns].mean() * len(games)
        control_indexes.update(controls.index)

    for label, totals in [("Game hunt dates", actual), ("Matched non-game", baseline)]:
        comparison_rows.append(
            {
                "team": team,
                "comparison": label,
                "game_hunt_dates": len(game_days),
                "control_pool_dates": len(control_indexes),
                "applicants_per_reservation": (
                    totals["first_choice_applicants"]
                    / totals["reservations_available"]
                ),
                "ducks_per_hunt_day": totals["ducks"] / len(game_days),
                "hunters_per_hunt_day": totals["hunters"] / len(game_days),
                "ducks_per_hunter": totals["ducks"] / totals["hunters"],
            }
        )

comparison = pd.DataFrame(comparison_rows)
metrics = [
    ("applicants_per_reservation", "First-choice applicants\nper available reservation"),
    ("ducks_per_hunt_day", "Ducks per hunt day"),
    ("hunters_per_hunt_day", "Hunters per hunt day"),
    ("ducks_per_hunter", "Ducks per hunter"),
]

fig, axes = plt.subplots(1, 4, figsize=(21, 6))
x = np.arange(len(teams))
width = 0.36
for axis, (metric, title) in zip(axes, metrics):
    for offset, label, color in [
        (-width / 2, "Game hunt dates", "#2563a6"),
        (width / 2, "Matched non-game", "#94a3b8"),
    ]:
        values = (
            comparison.loc[comparison["comparison"] == label]
            .set_index("team")
            .loc[[team[0] for team in teams], metric]
        )
        bars = axis.bar(x + offset, values, width, label=label, color=color)
        axis.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    axis.set_title(title, fontweight="bold")
    axis.set_xticks(x, [team[0] for team in teams], rotation=15, ha="right")
    axis.grid(axis="y", alpha=0.2)
    axis.set_ylim(0, axis.get_ylim()[1] * 1.12)

axes[0].legend(frameon=False, loc="upper left")
fig.suptitle(
    "Football game dates vs same-season, same-weekday non-game hunt dates",
    fontsize=16,
    fontweight="bold",
)
fig.text(
    0.5,
    0.01,
    "Overlapping hunt dates: Oregon 28, Oregon State 18, Seahawks 27. "
    "Lottery metric uses first-choice applications, not successful draws.",
    ha="center",
    fontsize=10,
)
fig.tight_layout(rect=[0, 0.05, 1, 0.93])
fig.savefig(ARTIFACTS / "sports-schedule-comparison.png", dpi=180, bbox_inches="tight")
plt.show()

## Initial finding

After matching non-game dates by hunting season and weekday:

- **Oregon game dates:** reservation demand was 2.33 versus 1.74 applicants per available reservation; harvest was 2.05 versus 1.92 ducks per hunter.
- **Oregon State game dates:** demand was 3.04 versus 1.86; harvest was 2.10 versus 1.94 ducks per hunter.
- **Seahawks game dates:** demand was lower at 1.52 versus 1.72, while harvest was higher at 1.84 versus 1.61 ducks per hunter.

College game dates therefore coincide with higher application demand in this sample; Seahawks dates do not. All three coincide with higher harvest efficiency after the weekday match. That does **not** mean football improves hunting: season timing, weather, migration, and the limited number of matching dates remain possible explanations.

In [ ]:
display_columns = [
    "team",
    "comparison",
    "game_hunt_dates",
    "control_pool_dates",
    "applicants_per_reservation",
    "ducks_per_hunt_day",
    "hunters_per_hunt_day",
    "ducks_per_hunter",
]
comparison[display_columns].round(2)

## Schedule coverage and exact overlaps

The database includes regular season, conference championship, bowl/playoff, and NFL postseason games. Kickoff timestamps are converted to Pacific time before date matching; NFL preseason is excluded.

In [ ]:
coverage = con.sql("""
    SELECT team, football_season, COUNT(*) AS games
    FROM football_game
    GROUP BY team, football_season
    ORDER BY team, football_season
""").df()

overlaps = con.sql("""
    SELECT
        team,
        COUNT(*) AS games_on_hunt_dates,
        MIN(hunt_date) AS first_overlap,
        MAX(hunt_date) AS last_overlap
    FROM sports_on_hunt_date
    GROUP BY team
    ORDER BY team
""").df()

coverage.pivot(index="football_season", columns="team", values="games"), overlaps